In [ ]:
# imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# encontrar ficheiros

DATA_DIR = Path(".")

audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

print("Número de ficheiros encontrados:", len(audio_files))

for file in audio_files:
    print(file.name)

In [ ]:
# ver os dados que o ficheiro tem, linhas, colunas, features, embedding da voz...

first_file = audio_files[0]

print("Ficheiro escolhido:", first_file.name)

df_example = pd.read_pickle(first_file)

print("Dimensão do ficheiro:", df_example.shape)

df_example.head()

print("Número de colunas:", len(df_example.columns))

for i, col in enumerate(df_example.columns, start=1):
    print(i, "-", col)



In [ ]:
# verficar embedding de voz 

embedding_example = df_example["speak_embeddings"].dropna().iloc[0]

print("Tipo do embedding:", type(embedding_example))
print("Dimensão do embedding:", len(embedding_example))
print("Primeiros 10 valores:")
print(embedding_example[:10])

In [ ]:
# PARTE 2: carregar os 28 ficheiros e extrair dados

def extract_metadata_from_filename(file_path):
    filename = file_path.name.replace("_audio.pkl", "")
    
    candidate_part, date_part = filename.split("_vs_")
    
    candidate_1 = candidate_part.replace("_", " ")
    
    date_parts = date_part.split("_")
    month = date_parts[-2]
    day = int(date_parts[-1])
    candidate_2 = " ".join(date_parts[:-2])
    
    debate_name = f"{candidate_1} vs {candidate_2}"
    
    return {
        "source_file": file_path.name,
        "debate_name": debate_name,
        "candidate_1": candidate_1,
        "candidate_2": candidate_2,
        "month": month,
        "day": day
    }


In [ ]:
# testar a função 

extract_metadata_from_filename(first_file)

In [ ]:
# carregar todos os files

all_dfs = []

for file_path in audio_files:
    df = pd.read_pickle(file_path)
    metadata = extract_metadata_from_filename(file_path)
    
    for key, value in metadata.items():
        df[key] = value
    
    all_dfs.append(df)

audio_raw = pd.concat(all_dfs, ignore_index=True)

print("Dimensão final do dataframe:", audio_raw.shape)
print("Número de debates:", audio_raw["source_file"].nunique())
print("Número total de segmentos:", len(audio_raw))

audio_raw.head()

In [ ]:
# Parte 3: verificar qualidade dos dados, ou seja se têm todos a mesma estrutura, se há valores em falta ou impossíveis...

# verificar a estrutura do ficheiro

structure_report = []

for file_path in audio_files:
    df = pd.read_pickle(file_path)
    
    structure_report.append({
        "source_file": file_path.name,
        "n_rows": df.shape[0],
        "n_columns": df.shape[1],
        "columns": list(df.columns)
    })

structure_report = pd.DataFrame(structure_report)

print("Todos os ficheiros têm 33 colunas?")
print((structure_report["n_columns"] == 33).all())

structure_report[["source_file", "n_rows", "n_columns"]]




In [ ]:
# verificar embedding

audio_raw["embedding_dim"] = audio_raw["speak_embeddings"].apply(
    lambda x: len(x) if isinstance(x, np.ndarray) else np.nan
)

audio_raw["embedding_dim"].value_counts(dropna=False)


In [ ]:
audio_clean = audio_raw.copy()

print("Dimensão do dataframe usado na análise:", audio_clean.shape)

In [ ]:
audio_clean["segment_start"] = audio_clean["time stamp"]
audio_clean["segment_end"] = audio_clean["time stamp"] + audio_clean["duration"]

audio_clean[["debate_name", "segment_start", "duration", "segment_end"]].head()

In [ ]:
summary_by_debate = (
    audio_clean
    .groupby(["debate_name", "candidate_1", "candidate_2", "source_file"])
    .agg(
        n_segments=("duration", "count"),
        total_speech_duration_sec=("duration", "sum"),
        estimated_debate_duration_sec=("segment_end", "max"),
        mean_segment_duration_sec=("duration", "mean"),
        median_segment_duration_sec=("duration", "median"),
        max_segment_duration_sec=("duration", "max"),
        mean_speechrate=("speechrate", "mean"),
        mean_articulationrate=("articulationrate", "mean"),
        total_pauses=("npause", "sum"),
        mean_pauses=("npause", "mean"),
        mean_pitch=("meanF0Hz", "mean"),
        pitch_variability=("stdevF0Hz", "mean"),
        mean_HNR=("HNR", "mean"),
        mean_jitter=("localJitter", "mean"),
        mean_shimmer=("localShimmer", "mean")
    )
    .reset_index()
)

summary_by_debate["estimated_debate_duration_min"] = (
    summary_by_debate["estimated_debate_duration_sec"] / 60
)

summary_by_debate["total_speech_duration_min"] = (
    summary_by_debate["total_speech_duration_sec"] / 60
)

summary_by_debate["segments_per_minute"] = (
    summary_by_debate["n_segments"] / summary_by_debate["estimated_debate_duration_min"]
)

summary_by_debate["pauses_per_minute"] = (
    summary_by_debate["total_pauses"] / summary_by_debate["total_speech_duration_min"]
)

summary_by_debate = summary_by_debate.round(3)

summary_by_debate.head()

In [ ]:
# debates mais fragmentados

summary_by_debate[
    ["debate_name", "n_segments", "segments_per_minute", "mean_segment_duration_sec"]
].sort_values("segments_per_minute", ascending=False).head(10)

In [ ]:
# debates com intervenções mais longas

summary_by_debate[
    ["debate_name", "mean_segment_duration_sec", "median_segment_duration_sec", "max_segment_duration_sec"]
].sort_values("mean_segment_duration_sec", ascending=False).head(10)

In [ ]:
# debates mais acelerados 

summary_by_debate[
    ["debate_name", "mean_speechrate", "mean_articulationrate"]
].sort_values("mean_speechrate", ascending=False).head(10)


In [ ]:
# debates com mais pausas por minuto

summary_by_debate[
    ["debate_name", "pauses_per_minute", "mean_pauses", "total_pauses"]
].sort_values("pauses_per_minute", ascending=False).head(10)

In [ ]:
# debates com maior variação local 

summary_by_debate[
    ["debate_name", "mean_pitch", "pitch_variability", "mean_HNR", "mean_jitter", "mean_shimmer"]
].sort_values("pitch_variability", ascending=False).head(10)

In [ ]:
# PARTE DE ANÁlise : tabela por candidato 

candidate_rows = []

for _, row in summary_by_debate.iterrows():
    candidate_rows.append({
        "candidate": row["candidate_1"],
        "debate_name": row["debate_name"],
        "n_segments": row["n_segments"],
        "segments_per_minute": row["segments_per_minute"],
        "mean_segment_duration_sec": row["mean_segment_duration_sec"],
        "mean_speechrate": row["mean_speechrate"],
        "mean_articulationrate": row["mean_articulationrate"],
        "pauses_per_minute": row["pauses_per_minute"],
        "pitch_variability": row["pitch_variability"],
        "mean_HNR": row["mean_HNR"],
        "mean_jitter": row["mean_jitter"],
        "mean_shimmer": row["mean_shimmer"]
    })
    
    candidate_rows.append({
        "candidate": row["candidate_2"],
        "debate_name": row["debate_name"],
        "n_segments": row["n_segments"],
        "segments_per_minute": row["segments_per_minute"],
        "mean_segment_duration_sec": row["mean_segment_duration_sec"],
        "mean_speechrate": row["mean_speechrate"],
        "mean_articulationrate": row["mean_articulationrate"],
        "pauses_per_minute": row["pauses_per_minute"],
        "pitch_variability": row["pitch_variability"],
        "mean_HNR": row["mean_HNR"],
        "mean_jitter": row["mean_jitter"],
        "mean_shimmer": row["mean_shimmer"]
    })

candidate_debate_metrics = pd.DataFrame(candidate_rows)

candidate_summary = (
    candidate_debate_metrics
    .groupby("candidate")
    .agg(
        n_debates=("debate_name", "count"),
        avg_segments_per_minute=("segments_per_minute", "mean"),
        avg_segment_duration=("mean_segment_duration_sec", "mean"),
        avg_speechrate=("mean_speechrate", "mean"),
        avg_articulationrate=("mean_articulationrate", "mean"),
        avg_pauses_per_minute=("pauses_per_minute", "mean"),
        avg_pitch_variability=("pitch_variability", "mean"),
        avg_HNR=("mean_HNR", "mean"),
        avg_jitter=("mean_jitter", "mean"),
        avg_shimmer=("mean_shimmer", "mean")
    )
    .reset_index()
    .round(3)
)

candidate_summary

In [ ]:
# candidatos em debates mais acelerados 

candidate_summary.sort_values("avg_speechrate", ascending=False)

In [ ]:
# candidatos em debates mais fragmentados

candidate_summary.sort_values("avg_segments_per_minute", ascending=False)

In [ ]:
# candidatos em debates com intervenções mais longas

candidate_summary.sort_values("avg_segment_duration", ascending=False)

In [ ]:
# candidatos em debates com mais pausas

candidate_summary.sort_values("avg_pauses_per_minute", ascending=False)


In [ ]:
# GRÁFICOS ANÁLISE DOS DEBATES

def plot_barh(data, value_col, label_col, title, xlabel, top_n=10, ascending=True):
    plot_data = data.sort_values(value_col, ascending=ascending).head(top_n)

    plt.figure(figsize=(10, 6))
    plt.barh(plot_data[label_col], plot_data[value_col])
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# debates mais acelerados 

plot_barh(
    summary_by_debate,
    value_col="mean_speechrate",
    label_col="debate_name",
    title="Top 10 debates com maior speech rate médio",
    xlabel="Speech rate médio",
    top_n=10,
    ascending=False
)

In [ ]:
# debates mais fragmentados

plot_barh(
    summary_by_debate,
    value_col="segments_per_minute",
    label_col="debate_name",
    title="Top 10 debates mais fragmentados",
    xlabel="Segmentos por minuto",
    top_n=10,
    ascending=False
)

In [ ]:
# debates com intervenções mais longas

plot_barh(
    summary_by_debate,
    value_col="mean_segment_duration_sec",
    label_col="debate_name",
    title="Top 10 debates com intervenções médias mais longas",
    xlabel="Duração média dos segmentos (segundos)",
    top_n=10,
    ascending=False
)

In [ ]:
# debates com mais pausas

plot_barh(
    summary_by_debate,
    value_col="pauses_per_minute",
    label_col="debate_name",
    title="Top 10 debates com mais pausas por minuto",
    xlabel="Pausas por minuto de fala",
    top_n=10,
    ascending=False
)

In [ ]:
# Debates com maior variação vocal

plot_barh(
    summary_by_debate,
    value_col="pitch_variability",
    label_col="debate_name",
    title="Top 10 debates com maior variação de pitch",
    xlabel="Variação média de pitch",
    top_n=10,
    ascending=False
)

In [ ]:
# Relação entre ritmo e pausas

plt.figure(figsize=(9, 6))

plt.scatter(
    summary_by_debate["mean_speechrate"],
    summary_by_debate["pauses_per_minute"]
)

for _, row in summary_by_debate.iterrows():
    plt.text(
        row["mean_speechrate"],
        row["pauses_per_minute"],
        row["debate_name"],
        fontsize=7
    )

plt.title("Relação entre speech rate e pausas por minuto")
plt.xlabel("Speech rate médio")
plt.ylabel("Pausas por minuto")
plt.tight_layout()
plt.show()

In [ ]:
# Relação entre fragmentação e duração das intervenções

plt.figure(figsize=(9, 6))

plt.scatter(
    summary_by_debate["segments_per_minute"],
    summary_by_debate["mean_segment_duration_sec"]
)

for _, row in summary_by_debate.iterrows():
    plt.text(
        row["segments_per_minute"],
        row["mean_segment_duration_sec"],
        row["debate_name"],
        fontsize=7
    )

plt.title("Fragmentação vs duração média dos segmentos")
plt.xlabel("Segmentos por minuto")
plt.ylabel("Duração média dos segmentos (segundos)")
plt.tight_layout()
plt.show()

In [ ]:
# estimar numero de speakers por debate 

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

audio_speakers = audio_clean.copy()
audio_speakers["speaker_cluster"] = np.nan

for debate in audio_speakers["debate_name"].unique():
    mask = audio_speakers["debate_name"] == debate
    debate_df = audio_speakers.loc[mask].copy()
    
    embeddings = np.vstack(debate_df["speak_embeddings"].values)
    embeddings_norm = normalize(embeddings)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings_norm)
    
    audio_speakers.loc[mask, "speaker_cluster"] = labels

In [ ]:
speaker_summary = (
    audio_speakers
    .groupby(["debate_name", "speaker_cluster"])
    .agg(
        n_segments=("duration", "count"),
        total_speech_sec=("duration", "sum"),
        mean_segment_duration=("duration", "mean"),
        mean_speechrate=("speechrate", "mean")
    )
    .reset_index()
)

speaker_summary["total_speech_min"] = speaker_summary["total_speech_sec"] / 60

debate_totals = (
    speaker_summary
    .groupby("debate_name")["total_speech_sec"]
    .sum()
    .reset_index(name="debate_total_speech_sec")
)

speaker_summary = speaker_summary.merge(debate_totals, on="debate_name")

speaker_summary["speech_share"] = (
    speaker_summary["total_speech_sec"] / speaker_summary["debate_total_speech_sec"]
)

speaker_summary = speaker_summary.round(3)

speaker_summary.sort_values(["debate_name", "speech_share"], ascending=[True, False])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_DIR = Path(".")

audio_files = sorted(DATA_DIR.glob("*audio.pkl"))

print("Número de ficheiros audio.pkl encontrados:", len(audio_files))
audio_files[:5]

In [ ]:
df = pd.read_pickle(audio_files[0])

print("Shape:", df.shape)
display(df.head())
print(df.columns.tolist())

missing = df.isna().mean().sort_values(ascending=False) * 100
missing[missing > 0]

# Cada segmento de áudio foi convertido numa janela temporal com início e fim, permitindo localizar os momentos relevantes no vídeo.

df["end_time"] = df["time stamp"] + df["duration"]

def seconds_to_mmss(seconds):
    minutes = int(seconds // 60)
    sec = int(seconds % 60)
    return f"{minutes:02d}:{sec:02d}"

df["start_mmss"] = df["time stamp"].apply(seconds_to_mmss)
df["end_mmss"] = df["end_time"].apply(seconds_to_mmss)

df[["time stamp", "duration", "start_mmss", "end_mmss"]].head()

audio_features = [
    "duration",
    "meanF0Hz",
    "stdevF0Hz",
    "HNR",
    "localJitter",
    "localShimmer",
    "localdbShimmer",
    "npause",
    "speechrate",
    "articulationrate",
    "asd"
]

df[audio_features].describe().T

In [ ]:
# duração média dos segmentos de fala

plt.figure(figsize=(8, 4))
plt.hist(df["duration"], bins=30)
plt.xlabel("Duração do segmento (segundos)")
plt.ylabel("Frequência")
plt.title("Distribuição da duração dos segmentos de fala")
plt.grid(True)
plt.show()

In [ ]:
df.sort_values("duration", ascending=False)[
    ["start_mmss", "end_mmss", "duration", "npause", "speechrate", "articulationrate"]
].head(10)